In [1]:
import os, json, time
from pathlib import Path

import numpy as np

import tensorflow as tf
from tensorflow import keras
import pandas as pd

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, log_loss, confusion_matrix

CSV_PATH = Path("classification_with_c110_d110_errors_snr.csv")

OUT_DIR = Path.cwd() / "cnn_c110_truncation_out"
RUNS_DIR = OUT_DIR / "runs"
MODELS_DIR = OUT_DIR / "models"
for d in [OUT_DIR, RUNS_DIR, MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

HEARTBEAT_PATH = OUT_DIR / "HEARTBEAT.json"
LAST_EVENT_PATH = OUT_DIR / "LAST_EVENT.txt"
MANIFEST_PATH = OUT_DIR / "RUNNING_MANIFEST.json"

def hb(payload):
    payload = dict(payload); payload["time"] = time.ctime()
    HEARTBEAT_PATH.write_text(json.dumps(payload, indent=2))

def log_event(msg):
    LAST_EVENT_PATH.write_text(f"[{time.ctime()}] {msg}\n")

def _safe_html(s):
    return str(s).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")

class NotebookProgress:
    def __init__(self, total, desc="Progress"):
        self.total = max(int(total), 1)
        self.desc = desc
        self.n = 0
        self.postfix = ""
        self._mode = "text"
        self._label = None
        self._bar = None
        self._text_render(force=True)
        try:
            import ipywidgets as widgets
            from IPython.display import display
            self._label = widgets.HTML(value=self._html())
            self._bar = widgets.IntProgress(value=0, min=0, max=self.total)
            display(widgets.VBox([self._label, self._bar]))
            self._mode = "widget"
        except Exception:
            pass

    def _html(self):
        base = f"<b>{_safe_html(self.desc)}</b>: {self.n}/{self.total}"
        if self.postfix:
            base += f" | {_safe_html(self.postfix)}"
        return base

    def _text_render(self, force=False):
        if force:
            line = f"{self.desc}: {self.n}/{self.total}"
            if self.postfix:
                line += f" | {self.postfix}"
            print(line, flush=True)

    def set_postfix(self, text):
        self.postfix = str(text)
        if self._mode == "widget":
            self._label.value = self._html()

    def update(self, n=1):
        self.n = min(self.total, self.n + int(n))
        if self._mode == "widget":
            self._bar.value = self.n
            self._label.value = self._html()
        else:
            self._text_render(force=True)

    def close(self):
        if self._mode == "widget" and self._bar is not None:
            self._bar.bar_style = "success"
            self._label.value = self._html()
        else:
            self._text_render(force=True)

# ------------------------
# Experiment config
# ------------------------
K_MIN, K_MAX = 1, 55
K_LIST = list(range(K_MIN, K_MAX+1))

N_REPEATS = 100
TEST_FRAC = 0.15
VAL_FRAC = 0.15
VAL_FRAC_OF_REMAIN = VAL_FRAC / (1.0 - TEST_FRAC)

BASE_SEED = 1234
THR_GRID_SIZE = 800

EPOCHS = 260
PATIENCE = 35
BATCH_SIZE = 32
LR = 1e-3

# speed flags (safe)
tf.config.optimizer.set_jit(True)
try:
    tf.config.threading.set_inter_op_parallelism_threads(2)
    tf.config.threading.set_intra_op_parallelism_threads(6)
except Exception:
    pass

manifest = dict(
    started_at=time.ctime(),
    csv=str(CSV_PATH),
    out_dir=str(OUT_DIR.resolve()),
    plan=dict(K_MIN=K_MIN, K_MAX=K_MAX, N_REPEATS=N_REPEATS, TEST_FRAC=TEST_FRAC, VAL_FRAC=VAL_FRAC,
              EPOCHS=EPOCHS, PATIENCE=PATIENCE, BATCH_SIZE=BATCH_SIZE, LR=LR, THR_GRID_SIZE=THR_GRID_SIZE),
    tf_version=tf.__version__,
    gpus=[str(x) for x in tf.config.list_physical_devices("GPU")]
)
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2))
log_event("Manifest created.")
hb({"status":"ready"})

print("OUT_DIR:", OUT_DIR.resolve())
print("TF GPUs:", tf.config.list_physical_devices("GPU"))

OUT_DIR: /Users/kris/Desktop/Astrophysics/cnn_c110_truncation_out
TF GPUs: []


In [2]:
df = pd.read_csv(CSV_PATH)
if "y" not in df.columns:
    raise KeyError("CSV must contain column 'y' for labels")

df["y"] = df["y"].astype(int)

C110 = [f"c{i:03d}" for i in range(110)]
missing = [c for c in C110 if c not in df.columns]
if missing:
    raise KeyError(f"Missing c110 columns, e.g. {missing[:6]}")

BP = [f"c{i:03d}" for i in range(55)]          # c000..c054
RP = [f"c{i:03d}" for i in range(55, 110)]     # c055..c109

Xbp = df[BP].to_numpy(np.float32)
Xrp = df[RP].to_numpy(np.float32)
y_all = df["y"].to_numpy(int)

print("Xbp:", Xbp.shape, "Xrp:", Xrp.shape, "pos_rate:", float(y_all.mean()))

Xbp: (2815, 55) Xrp: (2815, 55) pos_rate: 0.19822380106571935


In [3]:
def make_splits(y, n_repeats, base_seed):
    splitter = StratifiedShuffleSplit(n_splits=n_repeats, test_size=TEST_FRAC, random_state=base_seed)
    idx_all = np.arange(len(y))
    splits = []
    for r, (trainval_rel, test_rel) in enumerate(splitter.split(np.zeros(len(y)), y)):
        idx_trainval = idx_all[trainval_rel]
        idx_test = idx_all[test_rel]

        y_tv = y[idx_trainval]
        splitter2 = StratifiedShuffleSplit(n_splits=1, test_size=VAL_FRAC_OF_REMAIN, random_state=base_seed + 1000 + r)
        tr_rel2, va_rel2 = next(splitter2.split(np.zeros(len(y_tv)), y_tv))

        idx_train = idx_trainval[tr_rel2]
        idx_val   = idx_trainval[va_rel2]
        splits.append(dict(repeat=r, train_idx=idx_train, val_idx=idx_val, test_idx=idx_test))
    return splits

splits = make_splits(y_all, N_REPEATS, BASE_SEED)
print("splits:", len(splits), "sizes:", {k: len(splits[0][k]) for k in ["train_idx","val_idx","test_idx"]})

splits: 100 sizes: {'train_idx': 1969, 'val_idx': 423, 'test_idx': 423}


In [4]:
def make_X_for_K(idx, K):
    """
    Keep first K BP coeffs and first K RP coeffs.
    Return X shape (N, K, 2) with channels: [BP, RP].
    """
    bp = Xbp[idx, :K]  # (N,K)
    rp = Xrp[idx, :K]  # (N,K)
    X = np.stack([bp, rp], axis=-1)  # (N,K,2)
    return X

# quick check
Xdemo = make_X_for_K(np.arange(5), 10)
print("demo shape:", Xdemo.shape)

demo shape: (5, 10, 2)


In [5]:
def prob_metrics(y_true, p):
    y_true = np.asarray(y_true).astype(int)
    p = np.asarray(p).astype(float)
    pc = np.clip(p, 1e-15, 1-1e-15)
    return dict(
        ROC_AUC=roc_auc_score(y_true, p) if len(np.unique(y_true))>1 else np.nan,
        PR_AUC=average_precision_score(y_true, p) if len(np.unique(y_true))>1 else np.nan,
        Brier=brier_score_loss(y_true, p),
        LogLoss=log_loss(y_true, pc, labels=[0,1]),
    )

def confusion_metrics(y_true, y_hat):
    tn, fp, fn, tp = confusion_matrix(y_true, y_hat, labels=[0,1]).ravel()
    sens = tp/(tp+fn) if (tp+fn) else np.nan
    spec = tn/(tn+fp) if (tn+fp) else np.nan
    prec = tp/(tp+fp) if (tp+fp) else np.nan
    return prec, sens, spec, tn, fp, fn, tp

def threshold_grid(p, n=800):
    lo, hi = float(np.min(p)), float(np.max(p))
    if not np.isfinite(lo) or not np.isfinite(hi) or lo == hi:
        return np.array([0.5])
    return np.linspace(lo, hi, n)

def choose_threshold(y_true, p, criterion="youden", n=800):
    best_thr, best_score = 0.5, -np.inf
    for thr in threshold_grid(p, n):
        yhat = (p >= thr).astype(int)
        prec, sens, spec, *_ = confusion_metrics(y_true, yhat)
        if criterion == "youden":
            score = (sens + spec - 1) if np.isfinite(sens) and np.isfinite(spec) else -np.inf
        elif criterion == "f1":
            score = (2*prec*sens/(prec+sens)) if (np.isfinite(prec) and np.isfinite(sens) and (prec+sens)>0) else -np.inf
        else:
            raise ValueError
        if score > best_score:
            best_score, best_thr = float(score), float(thr)
    return best_thr, float(best_score)

def evaluate(yv, pv, yt, pt):
    out = {f"test_{k}": float(v) for k,v in prob_metrics(yt, pt).items()}
    for crit in ["youden","f1"]:
        thr, sc = choose_threshold(yv, pv, crit, THR_GRID_SIZE)
        yhat = (pt >= thr).astype(int)
        prec, sens, spec, tn, fp, fn, tp = confusion_metrics(yt, yhat)
        out[f"{crit}_val_thr"] = thr
        out[f"{crit}_val_score"] = sc
        out[f"{crit}_test_Precision"] = float(prec)
        out[f"{crit}_test_Sensitivity"] = float(sens)
        out[f"{crit}_test_Specificity"] = float(spec)
        out[f"{crit}_test_tn"] = int(tn); out[f"{crit}_test_fp"] = int(fp)
        out[f"{crit}_test_fn"] = int(fn); out[f"{crit}_test_tp"] = int(tp)
    return out

In [6]:
def light_respectable_cnn(input_shape, base_ch=32, k=5, blocks=3, dropout=0.25, bn=True):
    inp = keras.layers.Input(shape=input_shape)
    x = inp
    ch = base_ch
    for _ in range(blocks):
        x = keras.layers.Conv1D(ch, k, padding="same")(x)
        if bn:
            x = keras.layers.BatchNormalization()(x)
        x = keras.layers.LeakyReLU()(x)
        x = keras.layers.MaxPool1D(pool_size=2, padding="same")(x)
        x = keras.layers.Dropout(dropout)(x)
        ch = min(ch*2, 128)

    x = keras.layers.GlobalAveragePooling1D()(x)
    x = keras.layers.Dense(64)(x)
    x = keras.layers.LeakyReLU()(x)
    x = keras.layers.Dropout(dropout)(x)
    out = keras.layers.Dense(1, activation="sigmoid")(x)

    model = keras.Model(inp, out)
    opt = keras.optimizers.Adam(learning_rate=LR)
    model.compile(optimizer=opt, loss="binary_crossentropy")
    return model

class EpochHeartbeat(keras.callbacks.Callback):
    def __init__(self, tag):
        super().__init__()
        self.tag = tag
    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        loss = float(logs.get("loss", np.nan))
        val_loss = float(logs.get("val_loss", np.nan))
        hb({
            "status": "training",
            "tag": self.tag,
            "epoch": int(epoch),
            "loss": loss,
            "val_loss": val_loss,
        })
        print(f"[{self.tag}] epoch {int(epoch)+1}/{EPOCHS} loss={loss:.5f} val_loss={val_loss:.5f}", flush=True)

def train_one(Xtr, ytr, Xva, yva, model_path: Path, tag: str):
    tf.keras.backend.clear_session()
    tf.random.set_seed(BASE_SEED + (hash(tag) % 1000000))

    model = light_respectable_cnn(input_shape=Xtr.shape[1:])
    best_model_path = model_path.with_name(f"{model_path.stem}.best.keras")

    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=False),
        keras.callbacks.ModelCheckpoint(str(best_model_path), monitor="val_loss",
                                        save_best_only=True, save_weights_only=False),
        EpochHeartbeat(tag),
    ]

    model.fit(Xtr, ytr, validation_data=(Xva,yva),
              epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=0, callbacks=callbacks)

    model = keras.models.load_model(best_model_path)
    model.save(model_path)
    return model

In [7]:
def run_id(K, rep):
    return f"K{int(K):02d}__r{int(rep):02d}"

def shard_path(K, rep):
    return RUNS_DIR / f"{run_id(K, rep)}.parquet"

def done(K, rep):
    return shard_path(K, rep).exists()

def save_row(row):
    pd.DataFrame([row]).to_parquet(RUNS_DIR / f"{row['run_id']}.parquet", index=False)

def load_all_rows():
    files = sorted(RUNS_DIR.glob("*.parquet"))
    if not files:
        return pd.DataFrame()
    return pd.concat([pd.read_parquet(p) for p in files], ignore_index=True)

log_event("Ready to run truncation sweep.")
hb({"status":"ready"})
print("RUNS_DIR:", RUNS_DIR.resolve())

RUNS_DIR: /Users/kris/Desktop/Astrophysics/cnn_c110_truncation_out/runs


In [8]:
total_runs = len(K_LIST) * N_REPEATS
resumed_runs = sum(1 for K in K_LIST for sp in splits if done(K, sp["repeat"]))
print(f"Resuming from saved runs: {resumed_runs}/{total_runs} ({resumed_runs/total_runs:.1%})")

run_bar = NotebookProgress(total_runs, desc="Runs (K,repeat)")
if resumed_runs:
    run_bar.update(resumed_runs)

for K in K_LIST:
    log_event(f"Starting K={K}")

    for sp in splits:
        rep = sp["repeat"]
        tag = f"K={K}|r={rep}"
        run_bar.set_postfix(tag)

        if done(K, rep):
            continue

        hb({"status":"run_start", "tag": tag})
        print(f"Starting run {tag}", flush=True)

        tr, va, te = sp["train_idx"], sp["val_idx"], sp["test_idx"]

        Xtr = make_X_for_K(tr, K); ytr = y_all[tr]
        Xva = make_X_for_K(va, K); yva = y_all[va]
        Xte = make_X_for_K(te, K); yte = y_all[te]

        model_path = MODELS_DIR / f"{run_id(K, rep)}.keras"

        t0 = time.time()
        model = train_one(Xtr, ytr, Xva, yva, model_path=model_path, tag=tag)

        pv = model.predict(Xva, batch_size=1024, verbose=0).reshape(-1)
        pt = model.predict(Xte, batch_size=1024, verbose=0).reshape(-1)

        met = evaluate(yva, pv, yte, pt)
        dt = time.time() - t0

        row = dict(
            run_id=run_id(K, rep),
            K=int(K),
            repeat=int(rep),
            model_path=str(model_path),
            runtime_s=float(dt),
            n_train=int(len(ytr)), n_val=int(len(yva)), n_test=int(len(yte)),
            test_pos_rate=float(np.mean(yte)),
            **met
        )
        save_row(row)

        hb({"status":"run_done", "tag": tag, "runtime_s": dt})
        print(f"Finished run {tag} in {dt/60.0:.2f} min", flush=True)
        run_bar.update(1)

run_bar.close()

log_event("ALL DONE.")
hb({"status":"completed"})
print("Done. Shards:", len(list(RUNS_DIR.glob('*.parquet'))), "Models:", len(list(MODELS_DIR.glob('*.keras'))))

Resuming from saved runs: 5500/5500 (100.0%)
Runs (K,repeat): 0/5500


Done. Shards: 5500 Models: 10857


In [9]:
from pathlib import Path
import pandas as pd

def _guess_out_dir():
    if "OUT_DIR" in globals():
        return Path(OUT_DIR)
    return Path.cwd() / "cnn_c110_truncation_out"

def _guess_runs_dir():
    if "RUNS_DIR" in globals():
        return Path(RUNS_DIR)
    return _guess_out_dir() / "runs"

if "load_all_rows" not in globals():
    def load_all_rows():
        runs_dir = _guess_runs_dir()
        files = sorted(runs_dir.glob("*.parquet")) if runs_dir.exists() else []
        if files:
            return pd.concat([pd.read_parquet(p) for p in files], ignore_index=True)
        raw_parquet = _guess_out_dir() / "truncation_cnn_raw.parquet"
        if raw_parquet.exists():
            return pd.read_parquet(raw_parquet)
        return pd.DataFrame()

res = load_all_rows()
print("rows:", res.shape)
display(res.head(5))

out_dir = _guess_out_dir()
out_dir.mkdir(parents=True, exist_ok=True)
if not res.empty:
    res.to_parquet(out_dir / "truncation_cnn_raw.parquet", index=False)
    res.to_csv(out_dir / "truncation_cnn_raw.csv", index=False)

# summary per K
metrics = [
    "runtime_s",
    "test_PR_AUC", "test_ROC_AUC", "test_LogLoss", "test_Brier",
    "youden_test_Precision", "youden_test_Sensitivity", "youden_test_Specificity", "youden_val_thr",
    "f1_test_Precision", "f1_test_Sensitivity", "f1_test_Specificity", "f1_val_thr",
]
if res.empty:
    print("No completed runs yet; summary file not written.")
else:
    metrics_present = [m for m in metrics if m in res.columns]
    missing_metrics = [m for m in metrics if m not in res.columns]
    if missing_metrics:
        print("Warning: missing metrics:", missing_metrics)
    if not metrics_present:
        print("No summary metrics available; summary file not written.")
    else:
        summ = res.groupby("K")[metrics_present].agg(["mean", "std"]).reset_index()
        summ.columns = ["K"] + [f"{c}_{s}" for c, s in summ.columns[1:]]
        summ = summ.sort_values("K").reset_index(drop=True)
        summ.to_csv(out_dir / "truncation_cnn_summary_byK.csv", index=False)
        display(summ.head(10))

rows: (5500, 31)


,run_id,K,repeat,model_path,runtime_s,n_train,n_val,n_test,test_pos_rate,test_ROC_AUC,...,youden_test_tp,f1_val_thr,f1_val_score,f1_test_Precision,f1_test_Sensitivity,f1_test_Specificity,f1_test_tn,f1_test_fp,f1_test_fn,f1_test_tp
0,K01__r00,1,0,/Users/kris/Desktop/Astrophysics/cnn_c110_trun...,7.345180,1969,423,423,0.198582,0.945849,...,74,0.407941,0.810811,0.890411,0.773810,0.976401,331,8,19,65
1,K01__r01,1,1,/Users/kris/Desktop/Astrophysics/cnn_c110_trun...,10.882726,1969,423,423,0.198582,0.939071,...,72,0.292863,0.842105,0.868421,0.785714,0.970501,329,10,18,66
2,K01__r02,1,2,/Users/kris/Desktop/Astrophysics/cnn_c110_trun...,13.421071,1969,423,423,0.198582,0.863920,...,58,0.579105,0.867925,0.888889,0.666667,0.979351,332,7,28,56
3,K01__r03,1,3,/Users/kris/Desktop/Astrophysics/cnn_c110_trun...,5.908201,1969,423,423,0.198582,0.886220,...,71,0.123637,0.865497,0.825581,0.845238,0.955752,324,15,13,71
4,K01__r04,1,4,/Users/kris/Desktop/Astrophysics/cnn_c110_trun...,4.850696,1969,423,423,0.198582,0.929520,...,73,0.176771,0.754491,0.747126,0.773810,0.935103,317,22,19,65


,K,runtime_s_mean,runtime_s_std,test_PR_AUC_mean,test_PR_AUC_std,test_ROC_AUC_mean,test_ROC_AUC_std,test_LogLoss_mean,test_LogLoss_std,test_Brier_mean,...,youden_val_thr_mean,youden_val_thr_std,f1_test_Precision_mean,f1_test_Precision_std,f1_test_Sensitivity_mean,f1_test_Sensitivity_std,f1_test_Specificity_mean,f1_test_Specificity_std,f1_val_thr_mean,f1_val_thr_std
0,1,11.975517,2.863736,0.853558,0.036932,0.913162,0.023131,0.241794,0.052597,0.064080,...,0.214131,0.112866,0.863477,0.052748,0.774524,0.053583,0.968673,0.015312,0.313126,0.143810
1,2,13.138885,3.348880,0.877025,0.030907,0.922609,0.021128,0.204489,0.027261,0.052052,...,0.213058,0.100011,0.894732,0.044964,0.785000,0.047734,0.976401,0.011725,0.361179,0.123342
2,3,16.605744,4.535422,0.880860,0.028352,0.925525,0.020659,0.202226,0.026635,0.051608,...,0.191685,0.088334,0.892347,0.043351,0.785595,0.049751,0.975811,0.011520,0.352721,0.158476
3,4,18.475888,5.151477,0.878610,0.029533,0.923407,0.021977,0.202318,0.026891,0.051516,...,0.199399,0.090238,0.897185,0.042175,0.783214,0.051637,0.977109,0.011038,0.352569,0.142041
4,5,21.749019,5.980166,0.879531,0.028341,0.924969,0.020165,0.202683,0.026617,0.051861,...,0.190864,0.084765,0.898676,0.041922,0.783810,0.050358,0.977463,0.010849,0.361692,0.148781
5,6,24.494487,6.630698,0.881604,0.028719,0.924557,0.021114,0.201222,0.027808,0.051261,...,0.182312,0.082485,0.896831,0.041205,0.788690,0.050490,0.976932,0.010585,0.358908,0.135882
6,7,25.393223,6.217741,0.882589,0.028938,0.925856,0.020945,0.201135,0.027362,0.051269,...,0.171441,0.081407,0.894403,0.044093,0.791071,0.050291,0.976165,0.011465,0.334163,0.137048
7,8,27.162645,6.278707,0.883502,0.027687,0.927743,0.019774,0.200154,0.027659,0.051136,...,0.185090,0.088050,0.895502,0.041022,0.789405,0.052419,0.976519,0.010973,0.348756,0.149733
8,9,29.816734,6.858080,0.884832,0.027867,0.927938,0.019623,0.199449,0.027655,0.050939,...,0.168388,0.065676,0.890339,0.042721,0.792857,0.056476,0.975015,0.011493,0.337794,0.146047
9,10,31.323452,7.355988,0.884610,0.027915,0.928621,0.020040,0.200835,0.027140,0.051270,...,0.176938,0.064181,0.885608,0.044743,0.792500,0.050910,0.973894,0.012147,0.336258,0.151879


In [10]:
import plotly.graph_objects as go

def resampling_saddle_surface(metric="test_PR_AUC", n_sigma=4.0, n_x=160):
    if res.empty:
        print(f"Skipping {metric}: no rows in res")
        return
    if metric not in res.columns:
        print(f"Skipping {metric}: metric not found")
        return

    stats = (
        res[["K", metric]]
        .dropna()
        .groupby("K", as_index=False)
        .agg(mu=(metric, "mean"), sigma=(metric, "std"), n=(metric, "size"))
    )
    stats = stats[(stats["n"] >= 2) & np.isfinite(stats["sigma"]) & (stats["sigma"] > 0)].copy()
    if stats.empty:
        print(f"Skipping {metric}: not enough repeats with non-zero std")
        return

    x_min = float((stats["mu"] - n_sigma * stats["sigma"]).min())
    x_max = float((stats["mu"] + n_sigma * stats["sigma"]).max())
    x_grid = np.linspace(x_min, x_max, n_x)

    ks = stats["K"].to_numpy(float)
    mus = stats["mu"].to_numpy(float)
    sigs = stats["sigma"].to_numpy(float)

    X = np.repeat(ks[:, None], n_x, axis=1)
    Y = np.tile(x_grid, (len(ks), 1))
    Z = np.zeros_like(Y)

    for i, (mu, sigma) in enumerate(zip(mus, sigs)):
        t = (x_grid - mu) / sigma
        Z[i, :] = np.exp(-0.5 * t * t) / (sigma * np.sqrt(2.0 * np.pi))

    fig = go.Figure(data=[go.Surface(x=X, y=Y, z=Z, colorscale="Viridis")])
    fig.update_layout(
        title=f"Resampling saddle surface for {metric} (Gaussian from mean/std per K)",
        scene=dict(
            xaxis_title="K (coeffs per BP/RP)",
            yaxis_title=metric,
            zaxis_title="density (from repeats)",
            aspectmode="manual",
            aspectratio=dict(x=1.8, y=1.0, z=0.8),
        ),
        scene_camera=dict(eye=dict(x=2.2, y=1.3, z=0.9)),
        height=700,
        margin=dict(l=20, r=20, t=60, b=20),
    )
    fig.show()

resampling_saddle_surface("test_PR_AUC")
resampling_saddle_surface("test_LogLoss")
resampling_saddle_surface("youden_test_Sensitivity")
resampling_saddle_surface("f1_test_Precision")

In [11]:
import plotly.graph_objects as go

def resampling_saddle_surface_binned(metric="test_PR_AUC", n_bins=80, min_repeats=1):
    if res.empty:
        print(f"Skipping {metric}: no rows in res")
        return
    if metric not in res.columns:
        print(f"Skipping {metric}: metric not found")
        return

    df = res[["K", metric]].dropna().copy()
    if df.empty:
        print(f"Skipping {metric}: no finite values after dropna")
        return

    counts = df.groupby("K", as_index=False).size().rename(columns={"size": "n"})
    valid_k = counts.loc[counts["n"] >= min_repeats, "K"]
    df = df[df["K"].isin(valid_k)].copy()
    if df.empty:
        print(f"Skipping {metric}: no K groups with at least {min_repeats} repeats")
        return

    x_min = float(df[metric].min())
    x_max = float(df[metric].max())
    if not np.isfinite(x_min) or not np.isfinite(x_max):
        print(f"Skipping {metric}: non-finite metric range")
        return
    if x_max == x_min:
        eps = max(abs(x_min), 1.0) * 1e-6
        x_min -= eps
        x_max += eps

    bin_edges = np.linspace(x_min, x_max, n_bins + 1)
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

    ks = np.sort(df["K"].unique().astype(float))
    X = np.repeat(ks[:, None], n_bins, axis=1)
    Y = np.tile(bin_centers, (len(ks), 1))
    Z = np.zeros((len(ks), n_bins), dtype=float)

    for i, k in enumerate(ks):
        vals = df.loc[df["K"] == k, metric].to_numpy(float)
        hist, _ = np.histogram(vals, bins=bin_edges, density=True)
        Z[i, :] = hist

    fig = go.Figure(data=[go.Surface(x=X, y=Y, z=Z, colorscale="Viridis")])
    fig.update_layout(
        title=f"Resampling saddle surface for {metric} (raw histogram bins per K)",
        scene=dict(
            xaxis_title="K (coeffs per BP/RP)",
            yaxis_title=metric,
            zaxis_title="histogram density (raw bins)",
            aspectmode="manual",
            aspectratio=dict(x=1.8, y=1.0, z=0.8),
        ),
        scene_camera=dict(eye=dict(x=2.2, y=1.3, z=0.9)),
        height=700,
        margin=dict(l=20, r=20, t=60, b=20),
    )
    fig.show()

resampling_saddle_surface_binned("test_PR_AUC")
resampling_saddle_surface_binned("test_LogLoss")
#resampling_saddle_surface_binned("youden_test_Sensitivity")
#resampling_saddle_surface_binned("f1_test_Precision")

In [12]:
import plotly.express as px

def line_with_band(metric):
    if res.empty:
        print(f"Skipping {metric}: no rows in res")
        return
    if metric not in res.columns:
        print(f"Skipping {metric}: metric not found")
        return
    dfm = res.groupby("K", as_index=False).agg(
        mean=(metric, "mean"),
        std=(metric, "std"),
        q10=(metric, lambda s: s.quantile(0.10)),
        q25=(metric, lambda s: s.quantile(0.25)),
        q50=(metric, "median"),
        q75=(metric, lambda s: s.quantile(0.75)),
        q90=(metric, lambda s: s.quantile(0.90)),
    )
    dfm["lo_std"] = dfm["mean"] - dfm["std"]
    dfm["hi_std"] = dfm["mean"] + dfm["std"]

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=dfm["K"], y=dfm["q90"], mode="lines", line=dict(width=0), showlegend=False, hoverinfo="skip"))
    fig.add_trace(go.Scatter(x=dfm["K"], y=dfm["q10"], mode="lines", fill="tonexty",
                             line=dict(width=0), name="q10-q90", fillcolor="rgba(80,130,255,0.14)"))
    fig.add_trace(go.Scatter(x=dfm["K"], y=dfm["q75"], mode="lines", line=dict(width=0), showlegend=False, hoverinfo="skip"))
    fig.add_trace(go.Scatter(x=dfm["K"], y=dfm["q25"], mode="lines", fill="tonexty",
                             line=dict(width=0), name="q25-q75", fillcolor="rgba(80,130,255,0.26)"))
    fig.add_trace(go.Scatter(x=dfm["K"], y=dfm["mean"], mode="lines+markers", name="mean",
                             line=dict(color="rgb(32,74,180)", width=3), marker=dict(size=6, color="rgb(32,74,180)")))
    fig.update_layout(
        title=f"{metric} vs K",
        template="plotly_white",
        xaxis_title="K",
        yaxis_title=metric,
        height=560,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1.0),
    )
    fig.show()

for m in ["test_PR_AUC","test_ROC_AUC","test_LogLoss","test_Brier",
          "youden_test_Sensitivity","youden_test_Specificity","youden_test_Precision",
          "f1_test_Sensitivity","f1_test_Specificity","f1_test_Precision"]:
    line_with_band(m)

In [13]:
# heatmap of mean metric by K
from pathlib import Path
import pandas as pd
import plotly.express as px

def ensure_res():
    global res
    if "res" in globals() and isinstance(res, pd.DataFrame):
        return res

    if "load_all_rows" in globals():
        try:
            res = load_all_rows()
            return res
        except Exception as e:
            print(f"load_all_rows() failed: {e}")

    parquet_candidates = []
    if "OUT_DIR" in globals():
        parquet_candidates.append(Path(OUT_DIR) / "truncation_cnn_raw.parquet")
    parquet_candidates.append(Path.cwd() / "cnn_c110_truncation_out" / "truncation_cnn_raw.parquet")

    for p in parquet_candidates:
        if p.exists():
            res = pd.read_parquet(p)
            return res

    shard_dirs = []
    if "RUNS_DIR" in globals():
        shard_dirs.append(Path(RUNS_DIR))
    shard_dirs.append(Path.cwd() / "cnn_c110_truncation_out" / "runs")

    for d in shard_dirs:
        if not d.exists():
            continue
        files = sorted(d.glob("*.parquet"))
        if files:
            res = pd.concat([pd.read_parquet(p) for p in files], ignore_index=True)
            return res

    raise RuntimeError("Could not build `res`. Run setup/training cells or ensure result parquet files exist.")

def heatmap_means(metric):
    df = ensure_res()
    if df.empty:
        print(f"Skipping {metric}: res is empty")
        return
    if metric not in df.columns:
        print(f"Skipping {metric}: metric not found")
        return
    dfm = df.groupby("K", as_index=False)[metric].mean()
    fig = px.imshow(dfm.set_index("K").T, aspect="auto",
                    title=f"Heatmap: mean {metric} across repeats (K on x-axis)")
    fig.update_layout(height=280)
    fig.show()

for m in ["test_PR_AUC", "test_LogLoss", "youden_test_Sensitivity", "f1_test_Precision"]:
    heatmap_means(m)

# best K by PR-AUC mean
res_df = ensure_res()
if res_df.empty or "test_PR_AUC" not in res_df.columns:
    print("Cannot rank K: missing rows or test_PR_AUC")
else:
    bestK = res_df.groupby("K")["test_PR_AUC"].mean().sort_values(ascending=False).head(10)
    print("Top K by PR-AUC mean:")
    print(bestK)

Top K by PR-AUC mean:
K
28    0.894642
24    0.893504
38    0.893334
30    0.893299
36    0.892978
17    0.892823
54    0.892750
33    0.892693
18    0.892690
39    0.892653
Name: test_PR_AUC, dtype: float64


In [20]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

from scipy import stats
from statsmodels.nonparametric.smoothers_lowess import lowess

def summary_ci_by_K(df: pd.DataFrame, metric: str) -> pd.DataFrame:
    """
    Returns per-K summary with:
      mean, std, n, se, 95% CI
    """
    g = df.groupby("K")[metric]
    out = g.agg(["mean", "std", "count"]).reset_index()
    out = out.rename(columns={"count": "n"})
    out["se"] = out["std"] / np.sqrt(out["n"])
    out["tcrit"] = out["n"].apply(lambda n: stats.t.ppf(0.975, df=max(int(n)-1, 1)))
    out["ci95_lo"] = out["mean"] - out["tcrit"] * out["se"]
    out["ci95_hi"] = out["mean"] + out["tcrit"] * out["se"]
    return out

def lowess_fit_raw(df: pd.DataFrame, metric: str, frac: float = 0.22, n_points: int = 400) -> pd.DataFrame:
    """
    LOWESS fit on raw repeat-level points (K, metric).
    Returns smoothed values on a dense K grid for a continuous curve.
    """
    dat = df[["K", metric]].dropna().copy().sort_values("K")
    x = dat["K"].to_numpy()
    y = dat[metric].to_numpy()

    sm = lowess(y, x, frac=frac, it=0, return_sorted=True)
    sm_df = pd.DataFrame(sm, columns=["K", "smooth"]).groupby("K", as_index=False)["smooth"].mean()

    k_dense = np.linspace(dat["K"].min(), dat["K"].max(), n_points)
    smooth_dense = np.interp(k_dense, sm_df["K"], sm_df["smooth"])
    return pd.DataFrame({"K": k_dense, "smooth": smooth_dense})

def plot_metric_with_fit_and_ci(
    df: pd.DataFrame,
    metric: str,
    frac: float = 0.22,
    benchmark: float | None = None,
    title: str | None = None,
):
    summ = summary_ci_by_K(df, metric)
    fit = lowess_fit_raw(df, metric, frac=frac, n_points=400)

    fig = go.Figure()

    # 95% CI for mean at each K
    fig.add_trace(go.Scatter(
        x=summ["K"], y=summ["ci95_hi"],
        mode="lines", line=dict(width=0),
        showlegend=False, hoverinfo="skip"
    ))
    fig.add_trace(go.Scatter(
        x=summ["K"], y=summ["ci95_lo"],
        mode="lines", line=dict(width=0),
        fill="tonexty", fillcolor="rgba(124, 139, 161, 0.18)",
        name="mean 95% CI"
    ))

    # mean line (secondary)
    fig.add_trace(go.Scatter(
        x=summ["K"], y=summ["mean"],
        mode="lines+markers",
        name="mean",
        line=dict(color="#7C8BA1", width=2.2),
        marker=dict(size=5, color="#6A7890", opacity=0.9)
    ))

    # LOWESS fitted curve (main)
    fig.add_trace(go.Scatter(
        x=fit["K"], y=fit["smooth"],
        mode="lines",
        name="LOWESS fit",
        line=dict(color="#0E7490", width=4.2, shape="spline", smoothing=1.0)
    ))

    if benchmark is not None:
        fig.add_hline(
            y=float(benchmark),
            line_dash="dot",
            line_color="rgba(96, 104, 120, 0.9)",
            annotation_text=f"benchmark={benchmark:.3f}",
            annotation_position="top left"
        )

    fig.update_layout(
        template="none",
        title=title or f"{metric} vs K with mean CI and LOWESS fit",
        xaxis_title="K",
        yaxis_title=metric,
        height=560,
        paper_bgcolor="white",
        plot_bgcolor="white",
        font=dict(color="#2F3441"),
        legend=dict(
            orientation="h",
            x=0.01,
            y=0.99,
            bgcolor="rgba(255,255,255,0.55)",
            bordercolor="rgba(47,52,65,0.12)",
            borderwidth=1
        ),
        margin=dict(l=70, r=40, t=70, b=60),
    )
    fig.update_xaxes(
        showline=True,
        linewidth=1,
        linecolor="#9AA0AC",
        gridcolor="rgba(154,160,172,0.22)",
        zeroline=False
    )
    fig.update_yaxes(
        showline=True,
        linewidth=1,
        linecolor="#9AA0AC",
        gridcolor="rgba(154,160,172,0.22)",
        zeroline=False
    )
    fig.show()

    return summ, fit

# Main examples
summ_pr, fit_pr = plot_metric_with_fit_and_ci(
    res, "test_PR_AUC",
    frac=0.22,
    title="PR-AUC vs K - mean 95% CI with LOWESS fit"
)

summ_ll, fit_ll = plot_metric_with_fit_and_ci(
    res, "test_LogLoss",
    frac=0.22,
    title="Log Loss vs K - mean 95% CI with LOWESS fit"
)

summ_you_sens, fit_you_sens = plot_metric_with_fit_and_ci(
    res, "youden_test_Sensitivity",
    frac=0.22,
    title="Youden Sensitivity vs K - mean 95% CI with LOWESS fit"
)

summ_f1_prec, fit_f1_prec = plot_metric_with_fit_and_ci(
    res, "f1_test_Precision",
    frac=0.22,
    title="F1 Precision vs K - mean 95% CI with LOWESS fit"
)
